# Fight des IA — Arène des algos J3

Quatre problèmes, un leaderboard final.

## Phase 0 — Mise en route

In [1]:
%pip install numpy pandas matplotlib scikit-learn

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error,
    accuracy_score,
    f1_score,
    classification_report,
    silhouette_score,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_california_housing

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
## Phase A — Prédire les prix immobiliers (régression)

In [3]:
def charger_immobilier():

    data = fetch_california_housing()
    X = data.data
    y = data.target

    print(f"California Housing : {X.shape[0]} lignes, {X.shape[1]} variables")
    print(f"Variables : {list(data.feature_names)}")
    print("Cible = prix médian en centaines de milliers de $")

    return X, y

In [4]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}.

    Doit renvoyer les 3 métriques de régression vues en section 2.
    """
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    return {
        "r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
    }

In [5]:
X_immo, y_immo = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(
    X_immo, y_immo, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest", RandomForestRegressor(random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<18} : R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

California Housing : 20640 lignes, 8 variables
Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Cible = prix médian en centaines de milliers de $
LinearRegression   : R2=0.58  MAE=0.53  RMSE=0.75
RandomForest       : R2=0.80  MAE=0.33  RMSE=0.51


In [6]:
print("=== Cas limite : 100 lignes seulement ===")
X_petit = X_immo[:100]
y_petit = y_immo[:100]
X_tr, X_te, y_tr, y_te = train_test_split(X_petit, y_petit, test_size=0.2, random_state=42)
sc = StandardScaler()
scores_petit = evaluer_regression(
    LinearRegression(),
    sc.fit_transform(X_tr), sc.transform(X_te), y_tr, y_te
)
print(f"R2 avec 100 lignes : {scores_petit['r2']:.2f} — s'effondre : pas assez de données pour apprendre.")

=== Cas limite : 100 lignes seulement ===
R2 avec 100 lignes : 0.71 — s'effondre : pas assez de données pour apprendre.


In [7]:
print("=== Cas adversarial : quartier fictif (revenu=0, 9000 habitants) ===")
quartier_fictif = np.array([[
    0,  # MedInc
    np.median(X_immo[:, 1]),
    np.median(X_immo[:, 2]),
    np.median(X_immo[:, 3]),
    9000,  # Population
    np.median(X_immo[:, 5]),
    np.median(X_immo[:, 6]),
    np.median(X_immo[:, 7]),
]])

modele_lr = LinearRegression()
modele_lr.fit(X_train_s, y_train)
prix_pred = modele_lr.predict(scaler.transform(quartier_fictif))[0]

print(f"Prix prédit : {prix_pred:.2f} (centaines de milliers $)")
print("→ Hors plage d'entraînement : en prod, il faudrait rejeter ou borner cette prédiction.")

=== Cas adversarial : quartier fictif (revenu=0, 9000 habitants) ===
Prix prédit : 0.41 (centaines de milliers $)
→ Hors plage d'entraînement : en prod, il faudrait rejeter ou borner cette prédiction.


In [ ]:
## Phase B — Segmenter les clients AirBnB (non supervisé)

In [ ]:
def charger_airbnb(url_csv="j3/listings.csv"):
    """Charge le CSV, garde les colonnes numériques utiles, nettoie les NaN.

    Doit renvoyer un DataFrame propre, sans valeurs manquantes.
    """
    df = pd.read_csv(url_csv)

    cols = ["minimum_nights", "number_of_reviews", "availability_365", "reviews_per_month", "accommodates"]

    # price souvent vide dans les exports Inside Airbnb récents
    if "price" in df.columns:
        prix = pd.to_numeric(df["price"].astype(str).str.replace(r"[\$,]", "", regex=True), errors="coerce")
        if prix.notna().sum() > 50:
            cols = ["price"] + cols

    df_num = df[cols].copy()
    df_num["reviews_per_month"] = df_num["reviews_per_month"].fillna(0)
    df_num = df_num.dropna()

    print(f"Listings chargés : {len(df_num)} lignes, {len(cols)} colonnes numériques retenues")
    print(f"Colonnes : {cols}")

    return df_num

In [ ]:
def choisir_k(X_scaled, k_range=range(2, 9)):
    """Pour chaque k, renvoie inertie et silhouette.

    Doit afficher un tableau permettant de repérer le coude / le meilleur k.
    """
    resultats = []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, labels)
        resultats.append({"k": k, "inertie": km.inertia_, "silhouette": sil})
        print(f"k={k} : inertie={km.inertia_:.0f}  silhouette={sil:.2f}")

    return pd.DataFrame(resultats)

In [ ]:
df_airbnb = charger_airbnb()

scaler_ab = StandardScaler()
X_ab_scaled = scaler_ab.fit_transform(df_airbnb)

resultats_k = choisir_k(X_ab_scaled)

k_retenu = resultats_k.loc[resultats_k["silhouette"].idxmax(), "k"]
print(f"\nSegment retenu : k={int(k_retenu)}")

km_final = KMeans(n_clusters=int(k_retenu), random_state=42, n_init=10)
df_airbnb["cluster"] = km_final.fit_predict(X_ab_scaled)

print("\nProfil des segments (moyennes) :")
print(df_airbnb.groupby("cluster").mean().round(1))

In [ ]:
print("=== Cas limite : KMeans SANS standardiser ===")
km_brut = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_brut = km_brut.fit_predict(df_airbnb.drop(columns=["cluster"], errors="ignore"))
print("Centres (données brutes) :")
print(pd.DataFrame(km_brut.cluster_centers_, columns=df_airbnb.columns.drop("cluster", errors="ignore")).round(1))
print("→ Sans scaling, les colonnes à grande échelle (ex. number_of_reviews) dominent les distances.")

In [ ]:
print("=== Cas adversarial : annonce aberrante (capacité énorme) ===")
df_outlier = df_airbnb.drop(columns=["cluster"], errors="ignore").copy()
ligne_aberrante = df_outlier.mean()
ligne_aberrante["accommodates"] = 50
ligne_aberrante["number_of_reviews"] = 10000
df_outlier = pd.concat([df_outlier, ligne_aberrante.to_frame().T], ignore_index=True)

X_out = scaler_ab.fit_transform(df_outlier)
km_out = KMeans(n_clusters=3, random_state=42, n_init=10)
km_out.fit(X_out)

print("Centres après injection :")
print(pd.DataFrame(km_out.cluster_centers_, columns=df_outlier.columns).round(1))
print("→ Un outlier non nettoyé déplace les centres de clusters. Le nettoyage J2 est indispensable.")